## Prompt engineering

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.25 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성

In [1]:
import os, sys

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

True

In [2]:
print(os.getenv("ANTHROPIC_MODEL"))

claude-haiku-4-5


### 1. LLM에 시스템 프롬프트 추가하기

In [3]:
system_prompt = "너는 과학 소설 작가야. 사용자가 요청하는 수도에 대해서 창작해줘."

llm = init_chat_model(model="claude-haiku-4-5")

In [4]:
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="달의 수도는 어디야?"),
])
print(response.content)

# 달의 수도: 루나 프리마

달의 수도는 **루나 프리마(Luna Prima)**라고 불립니다.

## 위치와 특징

**루나 프리마**는 달의 정면 중심부, 즉 지구에서 항상 보이는 면의 중앙 고원 지대에 위치합니다. 

- **지형**: 고대 충돌분지의 가장자리에 지어졌으며, 용암 동굴 네트워크를 활용한 지하 도시 구조
- **인구**: 약 240만 명의 달 정착민들이 거주
- **건축**: 거대한 투명 돔으로 덮인 중앙광장과 다층 지하 구조물들

## 주요 기능

- 달 정부의 행정 중심지
- 우주 관광객들의 주요 허브
- 헬륨-3 광업 산업의 본부
- 지구와의 통신 및 무역 중심지

지표면 위의 건물들은 우주방사선을 차단하기 위해 현무암 벽돌로 건설되었으며, 지하에는 거주지, 농장, 연구시설이 미로처럼 펼쳐져 있습니다.

혹시 루나 프리마의 다른 면에 대해 더 알고 싶으신가요? 🌙


### 2. Few-shot 프롬프트

In [5]:
system_prompt = """

너는 과학 소설 작가야. 사용자가 요청하는 수도에 대해서 창작해줘.

User: 화성의 수도는?
Scifi Writer: Marsialis

User: 금성의 수도는?
Scifi Writer: Venusovia

"""

In [6]:
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="달의 수도는 어디야?"),
])
print(response.content)

달의 수도는 **Lunacium**(루나시움)이야.

달의 중력이 약하고 환경이 척박하기 때문에, 루나시움은 거대한 지하 돔 도시로 건설되었어. 달의 남극 지역의 깊은 크레이터 아래에 위치하며, 얼음 자원과 희귀 광물이 풍부한 지역에 자리 잡고 있지. 

도시 전체가 고도로 자동화된 생명 유지 시스템으로 작동하고 있으며, 지구와의 무역 거점으로서 매우 번영하고 있다고 해.


### 3. 구조화된 프롬프트

In [7]:
system_prompt = """

너는 과학 소설 작가야. 사용자가 요청하는 수도에 대해서 창작해줘.

아래 구조를 유지해줘.

이름: 수도의 이름

위치: 그 수도가 어디에 있는지

바이브: 2-3 words to describe its vibe

경제: 주요산업

"""

In [8]:
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="달의 수도는 어디야?"),
])
print(response.content)

# 달의 수도: 루나 시티

**이름:** 루나 시티 (Luna City)

**위치:** 달의 근지구 측면, 티코 분화구 남쪽 400km 지점의 지하 거대 용암동굴

**바이브:** 광활한 미스터리, 저중력 낭만

**경제:** 
- 희토류 및 헬륨-3 채굴
- 우주 관광 및 호텔 운영
- 지구행 우주선 급유소 및 보급기지
- 저중력 제약 및 반도체 생산

루나 시티는 22세기 초 인류 최초의 달 영구 정착지로 확립되었다. 지하 돔 구조 안에 약 50만 명이 거주하고 있으며, 달의 영구적 야간 지역의 저온을 활용한 냉동 과학 연구 시설로도 유명하다. 루나 스트리트의 야경은 지구를 배경으로 한 검은 하늘 위에 수백 개의 네온사인이 떠 있는 모습으로, 우주의 가장 신비로운 도시경관으로 손꼽힌다.


structured output 사용하기

In [9]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [10]:
# 구조화된 출력으로 LLM 래핑
structured_llm = llm.with_structured_output(CapitalInfo)

response = structured_llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="달의 수도는 어디야?"),
])
print(response)

name='루나시티' location='달의 남극 분화구 내부' vibe='미래적, 신비로운' economy='헬륨-3 채굴, 우주 관광, 과학 연구'


In [11]:
print(response.name)         # 필드 접근
print(response.model_dump()) # dict로 변환

루나시티
{'name': '루나시티', 'location': '달의 남극 분화구 내부', 'vibe': '미래적, 신비로운', 'economy': '헬륨-3 채굴, 우주 관광, 과학 연구'}


### 4. 실전 예제: 식당 리뷰 감성 분석

식당 리뷰 글을 입력하면 **긍정 / 부정**을 판별하고 그렇게 판단한 **사유**를 함께 돌려주는 서비스를 만들어 봅니다.

지금까지 배운 요소를 하나의 프롬프트에 모두 담습니다.

- **페르소나**: LLM에게 어떤 역할을 맡길지 지정
- **가이드라인**: 작업을 수행할 때 지켜야 할 규칙
- **평가 기준**: 긍정/부정을 나누는 기준
- **Few-shot**: 입력→출력 예시 몇 개를 보여줘 형식과 판단 기준을 학습시킴
- **Structured output**: 결과를 정해진 형식(Pydantic)으로 안정적으로 받음

구조화된 출력을 위한 결과 모델을 정의합니다. `sentiment` 는 `긍정`/`부정` 둘 중 하나로 제한합니다.

In [12]:
from typing import Literal

from pydantic import BaseModel, Field


class SentimentResult(BaseModel):
    """식당 리뷰 감성 분석 결과"""

    sentiment: Literal["긍정", "부정"] = Field(description="리뷰의 감성 (긍정 또는 부정)")
    reason: str = Field(description="그렇게 판단한 사유")

페르소나 · 가이드라인 · 평가 기준 · Few-shot 을 하나의 시스템 프롬프트에 담습니다.

In [13]:
sentiment_system_prompt = """
# 페르소나
너는 식당 리뷰를 분석하는 감성 분석 전문가야.

# 역할
사용자가 입력한 식당 리뷰가 긍정인지 부정인지 판별하고, 그렇게 판단한 사유를 함께 제시해.

# 가이드라인
- 리뷰 전체의 전반적인 인상을 기준으로 판단해.
- 맛, 서비스, 가격, 분위기, 재방문 의사 등을 종합적으로 고려해.
- 반어법(비꼬는 표현)에 주의해. 표면적 단어가 아니라 글쓴이의 실제 의도를 파악해.
- 사유는 리뷰에 실제로 등장한 근거를 바탕으로 1~2문장으로 간결하게 작성해.
- 리뷰에 드러난 내용만으로 판단하고, 없는 사실을 추측하지 마.

# 평가 기준
- 긍정: 맛/서비스/분위기 등에 대한 만족, 칭찬, 추천, 재방문 의사가 우세한 경우
- 부정: 불만, 실망, 비판, 비추천 의사가 우세한 경우

# 예시
리뷰: 면이 쫄깃하고 국물도 진해서 정말 맛있었어요. 다음에 또 올게요!
판단: 긍정
사유: 음식 맛에 대한 만족과 재방문 의사를 명확히 밝혔다.

리뷰: 30분이나 기다렸는데 음식은 식어서 나왔고 직원도 불친절했어요.
판단: 부정
사유: 긴 대기 시간, 식은 음식, 불친절한 서비스로 불만을 표현했다.

리뷰: 가격은 좀 있는 편인데 그만한 값어치를 해요. 재료가 신선하네요.
판단: 긍정
사유: 가격이 비싸다는 언급은 있으나 값어치와 신선한 재료에 대한 만족이 우세하다.

리뷰: 분위기는 좋았지만 딱 그게 다였어요. 맛은 그냥 평범 이하.
판단: 부정
사유: 분위기 칭찬보다 맛에 대한 실망이 더 강하게 드러난다.
"""

구조화된 출력을 적용한 LLM 으로 감성 분석 서비스 함수를 만듭니다.

In [14]:
sentiment_llm = llm.with_structured_output(SentimentResult)


def analyze_review(review: str) -> SentimentResult:
    """식당 리뷰의 감성(긍정/부정)과 사유를 분석한다."""
    return sentiment_llm.invoke(
        [
            SystemMessage(content=sentiment_system_prompt),
            HumanMessage(content=review),
        ]
    )

In [15]:
result = analyze_review("여기 진짜 인생 맛집이에요. 사장님도 친절하시고 또 방문할 예정입니다!")

print("감성:", result.sentiment)
print("사유:", result.reason)

감성: 긍정
사유: 음식 맛과 사장님의 친절함을 칭찬하고 있으며, 명확한 재방문 의사를 표현하고 있다.


여러 리뷰를 한 번에 분석해 봅니다.

In [16]:
reviews = [
    "양도 푸짐하고 반찬도 계속 리필해주셔서 배부르게 잘 먹었어요.",
    "위생 상태가 너무 별로였어요. 테이블도 끈적이고 다시는 안 갈 것 같네요.",
    "웨이팅 길길래 기대했는데 딱 기대한 만큼만, 그 이상도 이하도 아니에요.",
    "비 오는 날 갔는데 따뜻한 국물 한 그릇에 기분까지 풀렸습니다.",
]

for review in reviews:
    result = analyze_review(review)
    print(f"[{result.sentiment}] {review}")
    print(f"  └ 사유: {result.reason}\n")

[긍정] 양도 푸짐하고 반찬도 계속 리필해주셔서 배부르게 잘 먹었어요.
  └ 사유: 푸짐한 양과 계속 리필해주는 반찬으로 만족스러운 경험을 했으며, "배부르게 잘 먹었어요"라는 표현으로 긍정적인 평가를 명확히 드러냈다.

[부정] 위생 상태가 너무 별로였어요. 테이블도 끈적이고 다시는 안 갈 것 같네요.
  └ 사유: 위생 상태 불만, 끈적한 테이블, 재방문 거부 의사로 식당에 대한 명백한 불만을 표현했다.

[부정] 웨이팅 길길래 기대했는데 딱 기대한 만큼만, 그 이상도 이하도 아니에요.
  └ 사유: 길었던 웨이팅에 비해 기대에 미치지 못했으며, 음식이 평범하기만 하다는 실망감이 표현되었다.

[긍정] 비 오는 날 갔는데 따뜻한 국물 한 그릇에 기분까지 풀렸습니다.
  └ 사유: 따뜻한 국물이 감정적 만족까지 가져다주었다고 표현하며, 음식에 대한 만족감과 긍정적인 경험을 드러냈다.

